In [1]:
from matplotlib import pyplot as plt
%matplotlib inline
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import numpy as np

In [2]:
df = pd.read_csv('./heart.csv')
print(df.columns)
print(df.isnull().sum())
df.head()

Index(['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS',
       'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope',
       'HeartDisease'],
      dtype='str')
Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
dtype: int64


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [3]:
def calc_zscore_and_remove(
    df: pd.DataFrame,
    columns: list[str],
    threshold: float = 3.0,
) -> pd.DataFrame:
    """Return a copy of df without rows containing z-score outliers."""
    if threshold <= 0:
        raise ValueError("threshold must be greater than zero")

    selected = df.loc[:, columns]
    standard_deviations = selected.std(ddof=0).replace(0, np.nan)
    z_scores = (selected - selected.mean()) / standard_deviations
    outlier_rows = z_scores.abs().gt(threshold).any(axis=1)

    return df.loc[~outlier_rows].copy()

In [4]:
def encoding(df: pd.DataFrame) -> pd.DataFrame:
    df_encoded = pd.get_dummies(df, drop_first=True, dtype=int)
    return df_encoded

In [5]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

def ensemble_PCA(
    X: pd.DataFrame,
    y: np.ndarray,
    test_size: float = 0.2,
    random_state: int = 42,
) -> pd.DataFrame:
    """Compare binary classifiers after standard scaling and PCA."""
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y,
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    pca = PCA(n_components=0.95, random_state=random_state)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)

    models = {
        "SVM": SVC(random_state=random_state),
        "Logistic Regression": LogisticRegression(
            max_iter=1_000,
            random_state=random_state,
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=200,
            random_state=random_state,
        ),
    }

    results = []

    for model_name, model in models.items():
        model.fit(X_train_pca, y_train)
        y_pred = model.predict(X_test_pca)

        report = classification_report(
            y_test,
            y_pred,
            output_dict=True,
            zero_division=0,
        )

        results.append(
            {
                "model": model_name,
                "accuracy": report["accuracy"],
                "precision": report["macro avg"]["precision"],
                "recall": report["macro avg"]["recall"],
                "f1_score": report["macro avg"]["f1-score"],
                "pca_components": pca.n_components_,
            }
        )

    return (
        pd.DataFrame(results)
        .set_index("model")
        .sort_values("f1_score", ascending=False)
    )

In [6]:
TARGET_COLUMN = "HeartDisease"
OUTLIER_COLUMNS = ["Age", "RestingBP", "Cholesterol", "MaxHR"]

X = df.drop(columns=[TARGET_COLUMN]).copy()
y = df[TARGET_COLUMN].copy()

# Remove outliers, then retain the matching target values by index.
X_clean = calc_zscore_and_remove(
    X, columns=OUTLIER_COLUMNS, threshold=3.0
)
y_clean = y.loc[X_clean.index]

X_processed = encoding(X_clean)
assert X_processed.index.equals(y_clean.index)

model_results = ensemble_PCA(X_processed, y_clean.to_numpy())
print(model_results)

                     accuracy  precision    recall  f1_score  pca_components
model                                                                       
Random Forest        0.906593   0.908658  0.902927  0.905077              13
SVM                  0.901099   0.903923  0.896829  0.899349              13
Logistic Regression  0.901099   0.903923  0.896829  0.899349              13
